This notebook creates a "synthetic" (simulated) training environment to build a machine learning model that predicts biomass based on satellite data.

Here is the step-by-step breakdown of what the code does in plain English:

1. Data Simulation
For example purposes, we generate a "fake" dataset meant to demonstrate robustraster's machine learning capabilities.

It generates 5,000 rows of random numbers that look like raw data from a Landsat 8 satellite (ranging from 1 to 65,455).
It applies scaling factors and offsets (multiplying by 0.0000275 and subtracting 0.2) to turn those raw values into "Surface Reflectance" values.
It generates 5,000 random "biomass" values (randing from 0 to 300 megagrams per hectare) to serve as the ground truth the model needs to learn from.

2. Organizing the Data
It takes those random numbers and organizes them into a clean table (a DataFrame) with labels for the seven spectral bands (SR_B1 through SR_B7) and a final column for Biomass.

3. Training
The notebook then initializes a Gradient Boosting Regressor to train the model.

4. Saving the Result
Finally, once the model is trained, the notebook uses a tool called joblib to "pickle" (save) the model into a file called biomass_model.pkl.

We will use this biomass_model.pkl as input into the ml_example.ipynb.

In [ ]:
# Import NumPy for numerical array operations and random data generation
import numpy as np
# Import Pandas for structured tabular data manipulation
import pandas as pd
# Import the Gradient Boosting Regressor from scikit-learn's ensemble module
from sklearn.ensemble import GradientBoostingRegressor
# Import joblib for efficient serialization of trained model objects
import joblib
# Import os for operating system path utilities
import os

def create_and_train_model(output_path="biomass_model.pkl", n_samples=5000):
    """Generates synthetic Landsat 8 surface reflectance data, trains a
    Gradient Boosting regression model to predict biomass, and saves
    the trained model to disk as a serialized pickle file."""

    # Log the number of synthetic samples being generated
    print(f"Generating random dataset with {n_samples} samples...")

    # Generate a 2D array of random integers simulating raw Landsat 8
    # surface reflectance digital numbers (DN) across 7 spectral bands.
    # Valid DN range for Landsat 8 Collection 2 Level-2 is 1 to 65455.
    # Shape: (n_samples, 7) — one column per spectral band.
    raw_sr = np.random.uniform(1, 65455, size=(n_samples, 7))

    # Convert raw digital numbers to physical surface reflectance values
    # using the Landsat 8 Collection 2 Level-2 scaling formula:
    #   Reflectance = (DN × 0.0000275) + (−0.2)
    # This yields reflectance values in the approximate range of [−0.2, 1.6].
    scaled_sr = (raw_sr * 0.0000275) + (-0.2)

    # Generate random target biomass values uniformly distributed
    # between 0 and 300 (representing biomass in Mg/ha or equivalent units).
    # These serve as the dependent variable for model training.
    biomass = np.random.uniform(0, 300, size=n_samples)

    # Define the column names corresponding to the 7 Landsat 8
    # surface reflectance bands: Coastal Aerosol (B1), Blue (B2),
    # Green (B3), Red (B4), NIR (B5), SWIR-1 (B6), SWIR-2 (B7).
    bands = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']

    # Construct a Pandas DataFrame from the scaled reflectance array,
    # assigning each column the corresponding Landsat band name.
    df = pd.DataFrame(scaled_sr, columns=bands)

    # Append the target biomass values as a new column in the DataFrame.
    df['biomass'] = biomass

    # Log that model training is beginning
    print("Training Gradient Boosting Regressor (this may take a moment)...")

    # Instantiate a Gradient Boosting Regressor with default hyperparameters.
    # random_state=42 ensures reproducible results across runs.
    model = GradientBoostingRegressor(random_state=42)

    # Fit the model using the 7 spectral bands as predictor variables (X)
    # and the biomass column as the response variable (y).
    model.fit(df[bands], df['biomass'])

    # Serialize the trained model object to a pickle file on disk
    # using joblib, which is optimized for large NumPy arrays.
    joblib.dump(model, output_path)

    # Confirm the absolute file path where the model was saved
    print(f"Model successfully saved to: {os.path.abspath(output_path)}")
    # Inform the user that the exported model is ready for prediction use
    print("You can now use this pickle file in your Landsat ML prediction demo.")

# Execute the training function with default parameters
# (5000 samples, output to 'biomass_model.pkl')
create_and_train_model()
